# 07 — Gold: Resumo de Data Quality (`dq_monitoring_logs` → Gold)

Este notebook lê a tabela compartilhada `squad1/dq_monitoring_logs` (alimentada por **todas as duplas** do squad1, não só a nossa) e produz duas tabelas Gold, gravadas em dois destinos ao mesmo tempo (dual-sink: Delta no ADLS e Azure SQL Server). Diferente dos notebooks anteriores, este não gera novas regras de qualidade — ele agrega o que já foi medido.

## Regras/KPIs gerados por este notebook

- **`gold_dq_resumo_por_regra`** — 1 linha por combinação de (`tabela`, `regra`, `dia`), com `pct_falha` = % de registros que falharam aquela regra naquele dia. Serve para acompanhar a evolução de uma regra específica ao longo do tempo.
- **`gold_dq_resumo_por_tabela`** — 1 linha por combinação de (`tabela`, `hora`), com `pct_registros_limpos` = 100 − (% de checagens de regra que falharam naquela hora). É uma **aproximação**: mede "% de regras que passaram", não "% de linhas que passaram em todas as regras ao mesmo tempo" — o log não guarda qual linha específica falhou em qual regra, só a contagem agregada por execução.

Antes de agregar, o notebook corrige um bug real encontrado nos dados: algumas duplas gravam a severidade como `"Crítica"` (com acento) e outras como `"Critica"` (sem acento), o que quebraria qualquer agregação por severidade se não fosse normalizado. Isso ainda precisa ser alinhado com o squad/tech lead na fonte — aqui só é corrigido para fins de cálculo do Gold.

Ambas as tabelas são recalculadas do zero a cada execução (`overwrite`), então qualquer dupla pode rodar este notebook sem risco de corromper o resultado — desde que a fonte (`dq_monitoring_logs`) não tenha mudado de formato.

In [0]:
# ============================================================
# CÉLULA 1 — SETUP (ADLS + SQL SERVER)
# ============================================================

import uuid
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# --- Credenciais ADLS (igual sempre) ---
adls_client_id = dbutils.secrets.get(scope="internship-squad1", key="adls-client-id")
adls_tenant_id = dbutils.secrets.get(scope="internship-squad1", key="adls-tenant-id")
adls_client_secret = dbutils.secrets.get(scope="internship-squad1", key="adls-client-secret")
adls_storage_account = dbutils.secrets.get(scope="internship-squad1", key="adls-storage-account")

adls_options = {
    f"fs.azure.account.auth.type.{adls_storage_account}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{adls_storage_account}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{adls_storage_account}.dfs.core.windows.net": adls_client_id,
    f"fs.azure.account.oauth2.client.secret.{adls_storage_account}.dfs.core.windows.net": adls_client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{adls_storage_account}.dfs.core.windows.net": f"https://login.microsoftonline.com/{adls_tenant_id}/oauth2/token",
}

# --- Credenciais SQL Server (novas, pro dual-sink) ---
sql_host = dbutils.secrets.get(scope="internship-squad1", key="sql-host")
sql_database = dbutils.secrets.get(scope="internship-squad1", key="sql-database")
sql_user = dbutils.secrets.get(scope="internship-squad1", key="sql-username")
sql_password = dbutils.secrets.get(scope="internship-squad1", key="sql-password")

base_squad1 = f"abfss://squad1@{adls_storage_account}.dfs.core.windows.net"

caminhos = {
    "dq_logs": f"{base_squad1}/dq_monitoring_logs",
    "gold_resumo_por_regra": f"{base_squad1}/gold/gold_dq_resumo_por_regra",
    "gold_resumo_por_tabela": f"{base_squad1}/gold/gold_dq_resumo_por_tabela",
}

print("Setup concluído (ADLS + SQL Server).")
for k, v in caminhos.items():
    print(f"  {k}: {v}")

## Etapa 1 — Setup (ADLS + SQL Server)

Credenciais OAuth para o ADLS e, além disso, as credenciais do Azure SQL Server (`sql-host`, `sql-database`, `sql-username`, `sql-password`), necessárias porque este notebook grava em dois destinos (dual-sink). Os caminhos do Gold ficam na raiz do container `squad1` (pasta `gold/`), não sob `dupla1/`, pelo mesmo motivo do `dq_monitoring_logs`: o resultado é um resumo de todas as duplas, não só desta.

In [0]:
# ============================================================
# CÉLULA 2 — LER LOG E NORMALIZAR SEVERIDADE
# ============================================================

df_logs = spark.read.format("delta").options(**adls_options).load(caminhos["dq_logs"])

print("Severidades distintas ANTES da normalização:")
df_logs.select("severidade").distinct().show(truncate=False)

# Remove acento e padroniza capitalização (resolve Crítica/Critica -> Critica)
df_logs_norm = df_logs.withColumn(
    "severidade_norm",
    F.when(F.lower(F.translate(F.col("severidade"), "áàâãéèêíïóôõöúçñ", "aaaaeeeiioooocn")) == "critica", "Critica")
     .when(F.lower(F.translate(F.col("severidade"), "áàâãéèêíïóôõöúçñ", "aaaaeeeiioooocn")) == "aviso", "Aviso")
     .otherwise(F.col("severidade"))
)

print("\nSeveridades distintas DEPOIS da normalização:")
df_logs_norm.select("severidade_norm").distinct().show(truncate=False)

print(f"\nTotal de linhas no log (todas as duplas): {df_logs_norm.count()}")

## Etapa 2 — Leitura do log e normalização de severidade

Lê a tabela compartilhada completa (todas as duplas) e corrige a inconsistência real encontrada entre `"Crítica"` (com acento) e `"Critica"` (sem acento), usando `F.translate` para remover acentos e comparação em minúsculas, remapeando ambas para o valor canônico `"Critica"`/`"Aviso"` numa nova coluna `severidade_norm`. Sem essa normalização, qualquer agregação por severidade contaria `"Crítica"` e `"Critica"` como categorias diferentes.

In [0]:
# ============================================================
# CÉLULA 3 — GOLD_DQ_RESUMO_POR_REGRA (% falha por regra por dia)
# ============================================================

df_com_data = df_logs_norm.withColumn("data_execucao", F.to_date("timestamp_execucao"))

# Pega só a execução mais recente de cada (tabela, regra, dia) — evita contar reprocessamentos repetidos
janela_dia = Window.partitionBy("tabela", "regra", "data_execucao").orderBy(F.col("timestamp_execucao").desc())
df_dedup_dia = (
    df_com_data
    .withColumn("rank_execucao", F.row_number().over(janela_dia))
    .filter(F.col("rank_execucao") == 1)
    .drop("rank_execucao")
)

df_gold_resumo_por_regra = (
    df_dedup_dia
    .withColumn(
        "pct_falha",
        F.when(F.col("qtd_registros_total") > 0,
               F.round((F.col("qtd_registros_falhos") / F.col("qtd_registros_total")) * 100.0, 2))
         .otherwise(F.lit(0.0))
    )
    .select(
        F.col("data_execucao").alias("data"),
        "tabela",
        "regra",
        F.col("severidade_norm").alias("severidade"),
        "status",
        "qtd_registros_falhos",
        "qtd_registros_total",
        "pct_falha",
    )
    .withColumn("gold_processed_at", F.current_timestamp())
)

total_linhas_regra = df_gold_resumo_por_regra.count()
print(f"gold_dq_resumo_por_regra: {total_linhas_regra} linhas (1 por tabela/regra/dia)")
display(df_gold_resumo_por_regra.orderBy("data", "tabela", "regra"))

## Etapa 3 — `gold_dq_resumo_por_regra`

Antes de agregar, deduplica por (`tabela`, `regra`, `dia`) pegando só a execução mais recente via `Window` + `row_number`, porque algumas duplas reprocessam a mesma regra várias vezes (uma delas teve 353 execuções) — sem essa deduplicação, o `pct_falha` ficaria distorcido por contagem repetida. Depois calcula `pct_falha = qtd_registros_falhos / qtd_registros_total * 100`.

In [0]:
# ============================================================
# CÉLULA 4 — GOLD_DQ_RESUMO_POR_TABELA (% registros limpos por tabela por hora)
# ============================================================

df_com_hora = df_logs_norm.withColumn("hora_execucao", F.date_trunc("hour", F.col("timestamp_execucao")))

# Pega só a execução mais recente de cada (tabela, regra, hora)
janela_hora = Window.partitionBy("tabela", "regra", "hora_execucao").orderBy(F.col("timestamp_execucao").desc())
df_dedup_hora = (
    df_com_hora
    .withColumn("rank_execucao", F.row_number().over(janela_hora))
    .filter(F.col("rank_execucao") == 1)
    .drop("rank_execucao")
)

df_gold_resumo_por_tabela = (
    df_dedup_hora
    .groupBy("tabela", "hora_execucao")
    .agg(
        F.sum("qtd_registros_falhos").alias("soma_falhas"),
        F.sum("qtd_registros_total").alias("soma_total"),
        F.countDistinct("regra").alias("qtd_regras_avaliadas"),
        F.sum(F.when(F.col("status") == "FAIL", 1).otherwise(0)).alias("qtd_regras_com_falha"),
    )
    .withColumn(
        "pct_falha_ponderada",
        F.when(F.col("soma_total") > 0, F.round((F.col("soma_falhas") / F.col("soma_total")) * 100.0, 2)).otherwise(F.lit(0.0))
    )
    .withColumn("pct_registros_limpos", F.round(100.0 - F.col("pct_falha_ponderada"), 2))
    .select(
        F.col("hora_execucao").alias("hora"),
        "tabela",
        "qtd_regras_avaliadas",
        "qtd_regras_com_falha",
        "soma_falhas",
        "soma_total",
        "pct_registros_limpos",
    )
    .withColumn("gold_processed_at", F.current_timestamp())
)

total_linhas_tabela = df_gold_resumo_por_tabela.count()
print(f"gold_dq_resumo_por_tabela: {total_linhas_tabela} linhas (1 por tabela/hora)")
display(df_gold_resumo_por_tabela.orderBy("hora", "tabela"))

## Etapa 4 — `gold_dq_resumo_por_tabela`

Mesma lógica de deduplicação, mas por (`tabela`, `regra`, `hora`). Depois agrega por (`tabela`, `hora`), somando falhas e totais de todas as regras daquela tabela naquela hora, para chegar em `pct_registros_limpos = 100 - pct_falha_ponderada`. Como explicado no cabeçalho, isso é uma aproximação por checagem de regra, não por linha.

In [0]:
# ============================================================
# CÉLULA 5 — SINK 1: GRAVAR EM DELTA (ADLS)
# ============================================================

print("Gravando gold_dq_resumo_por_regra (Delta)...")
(df_gold_resumo_por_regra.write
    .format("delta")
    .options(**adls_options)
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(caminhos["gold_resumo_por_regra"]))
print(f"  -> Salvo: {caminhos['gold_resumo_por_regra']}")

print("Gravando gold_dq_resumo_por_tabela (Delta)...")
(df_gold_resumo_por_tabela.write
    .format("delta")
    .options(**adls_options)
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(caminhos["gold_resumo_por_tabela"]))
print(f"  -> Salvo: {caminhos['gold_resumo_por_tabela']}")

## Etapa 5 — Sink 1: gravação em Delta (ADLS)

Grava as duas tabelas Gold em Delta, em modo `overwrite` com `overwriteSchema=true` — cada execução recalcula o resumo do zero a partir do estado atual do log compartilhado.

In [0]:
# ============================================================
# CÉLULA 6 — SINK 2: GRAVAR NO SQL SERVER
# ============================================================

def salvar_no_sql_server(df, nome_tabela: str):
    """
    Grava o DataFrame na tabela relacional do schema squad1 no Azure SQL Server.
    Tenta o conector nativo 'sqlserver' primeiro; cai pro pyodbc se falhar.
    """
    print(f"Iniciando persistência no SQL Server na tabela '{nome_tabela}'...")

    try:
        (df.write
            .format("sqlserver")
            .option("host", sql_host)
            .option("port", "1433")
            .option("user", sql_user)
            .option("password", sql_password)
            .option("database", sql_database)
            .option("dbtable", nome_tabela)
            .mode("overwrite")
            .save())
        print(f"  [Sucesso] Tabela '{nome_tabela}' gravada via conector nativo 'sqlserver'!")
        return
    except Exception as e_sql:
        print(f"  Aviso: Conector 'sqlserver' direto retornou: {e_sql}")

    print("  -> Ativando fallback via pyodbc...")
    import pyodbc
    import pandas as pd
    pdf = df.toPandas()

    conn_str = (
        f"DRIVER={{ODBC Driver 18 for SQL Server}};"
        f"SERVER={sql_host};DATABASE={sql_database};UID={sql_user};PWD={sql_password};"
        "Encrypt=yes;TrustServerCertificate=no;"
    )
    try:
        conn = pyodbc.connect(conn_str, timeout=30)
    except Exception:
        conn_str_legacy = (
            f"DRIVER={{ODBC Driver 17 for SQL Server}};"
            f"SERVER={sql_host};DATABASE={sql_database};UID={sql_user};PWD={sql_password};"
            "Encrypt=yes;TrustServerCertificate=no;"
        )
        conn = pyodbc.connect(conn_str_legacy, timeout=30)

    cursor = conn.cursor()
    cursor.fast_executemany = True

    type_map = {
        'object': 'NVARCHAR(MAX)',
        'int64': 'BIGINT',
        'int32': 'INT',
        'float64': 'FLOAT',
        'bool': 'BIT',
        'datetime64[ns]': 'DATETIME2',
    }

    cols_def = [f"[{col_name}] {type_map.get(str(dtype), 'NVARCHAR(MAX)')}" for col_name, dtype in pdf.dtypes.items()]

    schema_name, table_raw = nome_tabela.split('.') if '.' in nome_tabela else ('squad1', nome_tabela)

    cursor.execute(f"IF NOT EXISTS (SELECT * FROM sys.schemas WHERE name = '{schema_name}') EXEC('CREATE SCHEMA [{schema_name}]')")
    cursor.execute(f"IF OBJECT_ID('[{schema_name}].[{table_raw}]', 'U') IS NOT NULL DROP TABLE [{schema_name}].[{table_raw}]")
    cursor.execute(f"CREATE TABLE [{schema_name}].[{table_raw}] ({', '.join(cols_def)})")
    conn.commit()

    placeholders = ', '.join(['?'] * len(pdf.columns))
    insert_sql = f"INSERT INTO [{schema_name}].[{table_raw}] VALUES ({placeholders})"
    records = [
        tuple(None if pd.isna(val) else (int(val) if isinstance(val, bool) else val) for val in row)
        for row in pdf.values
    ]
    cursor.executemany(insert_sql, records)
    conn.commit()
    cursor.close()
    conn.close()
    print(f"  [Sucesso] Tabela '{nome_tabela}' gravada via fallback pyodbc ({len(pdf)} linhas)!")

salvar_no_sql_server(df_gold_resumo_por_regra, "squad1.gold_dq_resumo_por_regra")
salvar_no_sql_server(df_gold_resumo_por_tabela, "squad1.gold_dq_resumo_por_tabela")

## Etapa 6 — Sink 2: gravação no SQL Server

`salvar_no_sql_server` (adaptada da função do notebook Gold de outra dupla do squad2) tenta primeiro o conector nativo do Databricks (`format("sqlserver")`) e, se falhar, cai para `pyodbc` com `CREATE TABLE` dinâmico (inferido a partir dos tipos do pandas) e inserção em lote via `executemany`. Nesta execução o conector nativo funcionou nas duas gravações — o fallback pyodbc nunca precisou ser usado. As tabelas de destino usam o schema `squad1` (`squad1.gold_dq_resumo_por_regra`, `squad1.gold_dq_resumo_por_tabela`) — essa notação `schema.tabela` é do SQL Server, não tem relação com Unity Catalog.